In [ ]:
# Lab type: review
# Course: DS203 — Feature Engineering & Pipelines
# Lesson: Auditing AI-Generated Preprocessing Pipelines
# Task: The code below is correct and working — read each section, run it, then answer
#       the judgment questions in the markdown cells. No code stubs to fill in.

# Lab: Auditing AI-Generated Preprocessing Pipelines

This lab gives you three AI-generated pipeline snippets — one for each failure
pattern from Lesson 4. Each snippet runs without Python errors. Your job is to
apply the five-point protocol and answer the judgment questions.

For each section:
1. Read the code carefully before running it.
2. Run the cell.
3. Answer the question in the markdown cell below.

**Outputs are cleared.** Run each cell to generate results.

## Setup: install dependencies and build the dataset

In [ ]:
!pip install scikit-learn pandas numpy --quiet

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 800

tenure_days     = np.random.exponential(300, n).clip(30, 1000).round(1)
monthly_spend   = np.random.lognormal(4.5, 0.6, n).round(2)
support_tickets = np.random.poisson(2, n).clip(0, 15).astype(float)
days_since_login = np.random.exponential(15, n).clip(1, 120).round(1)
invoice_amount  = np.random.lognormal(5.0, 0.8, n).round(2)
plan   = np.random.choice(["starter", "pro", "enterprise"], n, p=[0.5, 0.35, 0.15])
region = np.random.choice(["NA", "EMEA", "APAC", "LATAM"], n, p=[0.4, 0.3, 0.2, 0.1])

logit = (-0.003 * tenure_days + 0.4 * support_tickets
         + 0.05 * days_since_login - 0.02 * monthly_spend
         + 0.5 * (plan == "starter"))
churn_prob = 1 / (1 + np.exp(-logit))
churned = (np.random.rand(n) < churn_prob).astype(int)

df = pd.DataFrame({
    "tenure_days":      tenure_days,
    "monthly_spend":    monthly_spend,
    "support_tickets":  support_tickets,
    "days_since_login": days_since_login,
    "invoice_amount":   invoice_amount,
    "plan":             plan,
    "region":           region,
    "churned":          churned,
})

print(f"Dataset: {df.shape[0]} rows, churn rate {df['churned'].mean():.1%}")

## Pattern 1: Fit-before-split

The code below was generated by an AI tool given the instruction:
*"Build a churn model with a logistic regression classifier and proper preprocessing."*

Read it carefully. Apply protocol items 1 and 2 before running.

In [ ]:
# AI-generated code — apply the five-point protocol
numeric_cols = ["tenure_days", "monthly_spend", "support_tickets",
                "days_since_login", "invoice_amount"]

X_full = df[numeric_cols]
y = df["churned"]

# Scale the features
scaler_p1 = StandardScaler()
X_scaled_p1 = scaler_p1.fit_transform(X_full)   # ← note: before split

X_train_p1, X_test_p1, y_train_p1, y_test_p1 = train_test_split(
    X_scaled_p1, y, test_size=0.2, random_state=42
)

model_p1 = LogisticRegression(max_iter=500)
model_p1.fit(X_train_p1, y_train_p1)
auc_p1 = roc_auc_score(y_test_p1, model_p1.predict_proba(X_test_p1)[:, 1])
print(f"AUC (Pattern 1 — fit-before-split): {auc_p1:.3f}")

# Honest comparison: scaler fitted only on training rows
X_raw = df[numeric_cols]
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, random_state=42)
scaler_honest = StandardScaler()
X_tr_s = scaler_honest.fit_transform(X_tr)
X_te_s = scaler_honest.transform(X_te)
model_honest = LogisticRegression(max_iter=500)
model_honest.fit(X_tr_s, y_tr)
auc_honest = roc_auc_score(y_te, model_honest.predict_proba(X_te_s)[:, 1])
print(f"AUC (honest):                        {auc_honest:.3f}")
print(f"Inflation from fit-before-split:     {auc_p1 - auc_honest:+.3f}")

**Protocol item 1 — Split location:** Does `train_test_split` happen before any `fit` call? Identify the exact line that fails this check.

*(Write your answer here.)*

**Protocol item 2 — Pipeline completeness:** The `StandardScaler` is not inside a `Pipeline`. What does this mean for the serialised model? If you saved `model_p1` with `joblib.dump`, what preprocessing step would be missing at inference time?

*(Write your answer here.)*

## Pattern 2: Scaler outside cross-validation

The code below was generated for cross-validation. The `Pipeline` is present —
but does it contain all the transformers that should be inside it?

In [ ]:
# AI-generated code — apply protocol item 3
numeric_cols = ["tenure_days", "monthly_spend", "support_tickets",
                "days_since_login", "invoice_amount"]

X_raw_p2 = df[numeric_cols]
y = df["churned"]

X_train_p2, X_test_p2, y_train_p2, y_test_p2 = train_test_split(
    X_raw_p2, y, test_size=0.2, random_state=42
)

# Scale before cross-validation
scaler_p2 = StandardScaler()
X_train_scaled_p2 = scaler_p2.fit_transform(X_train_p2)   # ← applied before CV

pipeline_p2 = Pipeline([
    ("model", LogisticRegression(max_iter=500)),   # scaler is missing from pipeline
])
cv_scores_p2 = cross_val_score(pipeline_p2, X_train_scaled_p2, y_train_p2, cv=5, scoring="roc_auc")
print(f"CV AUC (Pattern 2): {cv_scores_p2.mean():.3f} \u00b1 {cv_scores_p2.std():.3f}")

# Honest comparison: scaler inside the pipeline
pipeline_honest_p2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(max_iter=500)),
])
cv_scores_honest = cross_val_score(pipeline_honest_p2, X_raw_p2, y, cv=5, scoring="roc_auc")
print(f"CV AUC (honest):    {cv_scores_honest.mean():.3f} \u00b1 {cv_scores_honest.std():.3f}")

**Protocol item 3 — Cross-validation scope:** In the buggy code, `X_train_scaled_p2` has been scaled using statistics from all training rows. When `cross_val_score` uses 5 folds, which rows from `X_train_p2` contributed to the scaler's mean and std for each validation fold?

*(Write your answer here.)*

**Practical consequence:** In the output above, the mean CV AUC values are similar. Does the absence of a large AUC gap mean there is no problem? What risk does the gap *understate*?

*(Write your answer here.)*

## Pattern 3: Hard-coded column lists

The code below uses explicit column lists — the pattern AI tools always generate.
Run it, then add a new column to the input schema and observe what happens.

In [ ]:
# AI-generated code — apply protocol item 4
numeric_cols_p3    = ["tenure_days", "monthly_spend", "support_tickets"]
categorical_cols_p3 = ["plan", "region"]

numeric_pipe_p3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipe_p3 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor_p3 = ColumnTransformer([
    ("num", numeric_pipe_p3,      numeric_cols_p3),
    ("cat", categorical_pipe_p3,  categorical_cols_p3),
])  # remainder="drop" is the default — unlisted columns are silently discarded

pipeline_p3 = Pipeline([
    ("preprocessor", preprocessor_p3),
    ("model",        LogisticRegression(max_iter=500)),
])

X_p3 = df[numeric_cols_p3 + categorical_cols_p3]
y = df["churned"]
X_tr_p3, X_te_p3, y_tr_p3, y_te_p3 = train_test_split(X_p3, y, test_size=0.2, random_state=42)
pipeline_p3.fit(X_tr_p3, y_tr_p3)
auc_p3 = roc_auc_score(y_te_p3, pipeline_p3.predict_proba(X_te_p3)[:, 1])
print(f"AUC (baseline, 3 numeric features): {auc_p3:.3f}")

# Simulate schema change: a new numeric column 'days_since_login' is added at inference time
X_new_schema = X_te_p3.copy()
X_new_schema["days_since_login"] = df.loc[X_te_p3.index, "days_since_login"]

# Does the pipeline raise an error?
try:
    auc_new = roc_auc_score(y_te_p3, pipeline_p3.predict_proba(X_new_schema)[:, 1])
    print(f"AUC with new column present:        {auc_new:.3f}")
    print("No error was raised — days_since_login was silently dropped.")
except Exception as e:
    print(f"Error: {e}")

**Protocol item 4 — Column coverage:** `days_since_login` was added to the input DataFrame but produced no error and no change in AUC. What happened to it, and why is silent dropping more dangerous than an exception?

*(Write your answer here.)*

**Production scenario:** Imagine a data engineering team adds `account_age_days` to the feature table six months after the model is deployed. The pipeline has been running in production for three months since then. Describe the failure mode and how you would detect it.

*(Write your answer here.)*

## Applying the five-point protocol: full worked example

The cell below reproduces the complete AI-generated pipeline from Lesson 4.
Apply all five protocol items before running the cell — make a note of each
finding, then read the checklist questions after.

In [ ]:
# AI-generated pipeline — apply the five-point protocol
numeric_cols_fp    = ["tenure_days", "monthly_spend", "support_tickets",
                      "days_since_login", "invoice_amount"]
categorical_cols_fp = ["plan", "region"]

X_fp = df.drop("churned", axis=1)[numeric_cols_fp + categorical_cols_fp]
y_fp = df["churned"]

# Preprocessing (note placement)
scaler_fp = StandardScaler()
X_fp[numeric_cols_fp] = scaler_fp.fit_transform(X_fp[numeric_cols_fp])  # ← outside pipeline

numeric_pipe_fp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    # scaler is missing — already applied above
])
categorical_pipe_fp = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor_fp = ColumnTransformer([
    ("num", numeric_pipe_fp,      numeric_cols_fp),
    ("cat", categorical_pipe_fp,  categorical_cols_fp),
])
full_pipeline_fp = Pipeline([
    ("preprocessor", preprocessor_fp),
    ("model",        LogisticRegression(max_iter=500)),
])

X_train_fp, X_test_fp, y_train_fp, y_test_fp = train_test_split(
    X_fp, y_fp, test_size=0.2, random_state=42
)
full_pipeline_fp.fit(X_train_fp, y_train_fp)

cv_scores_fp = cross_val_score(
    full_pipeline_fp, X_train_fp, y_train_fp, cv=5, scoring="roc_auc"
)
print(f"CV AUC: {cv_scores_fp.mean():.3f}")

**Protocol item 1 — Split location:** Does `train_test_split` happen before any `fit` call? Answer pass/fail and cite the line number.

*(Write your answer here.)*

**Protocol item 2 — Pipeline completeness:** Is `StandardScaler` inside `full_pipeline_fp`? If not, what happens to `scaler_fp` when you call `joblib.dump(full_pipeline_fp, "model.pkl")`?

*(Write your answer here.)*

**Protocol item 3 — Cross-validation scope:** `full_pipeline_fp` is passed to `cross_val_score`. Does this mean the cross-validation is honest? Consider what happened to `X_fp` before the split.

*(Write your answer here.)*

**Protocol item 4 — Column coverage:** Are column lists hard-coded? What would happen if a third categorical column were added to the input schema?

*(Write your answer here.)*

**Protocol item 5 — Leakage audit:** The feature `days_since_login` is included. For a churn model predicting whether a customer will leave next month, answer the availability question: would `days_since_login` be available at prediction time for a live customer?

*(Write your answer here.)*

## Fixing the worked example

Items 1 and 2 of the protocol fail. The fix is small — two lines change.
Rewrite the cell above so that:
- `StandardScaler` is inside the numeric sub-pipeline
- The manual `scaler_fp.fit_transform` block is deleted
- Everything else stays the same

In [ ]:
# Rewrite the AI-generated example with items 1 and 2 fixed
# Keep the same column lists, same model, same CV setup — only move the scaler

numeric_cols_fix    = ["tenure_days", "monthly_spend", "support_tickets",
                       "days_since_login", "invoice_amount"]
categorical_cols_fix = ["plan", "region"]

X_fix = df[numeric_cols_fix + categorical_cols_fix].copy()
y_fix = df["churned"]

# your code here: rebuild with the scaler inside numeric_pipe_fix
# numeric_pipe_fix = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler",  StandardScaler()),   # scaler belongs here
# ])
# ... (rest of the pipeline)

pass

## Summary

> **Reflect on the five-point protocol. Answer each question in one sentence.**

1. Which protocol items (1–4) can be checked mechanically by reading the code top-to-bottom in under two minutes?
2. Which item cannot be checked mechanically, and what domain knowledge does it require?
3. After fixing items 1 and 2 in the worked example, which items remain open and require further investigation?
4. Why is the protocol ordered 1–5 rather than presented as an unordered checklist?